# 2. Leak-free feature engineering and modeling

Notebook 1 found that specific exact and nearby income/commute values carry signal the model
can't see on its own. This notebook builds that signal into a feature, trains two different
gradient-boosting models on it, and checks the result honestly with cross-validation.

**A note on scale:** the real submission used 10 cross-validation folds, 6,000 trees per model
(with early stopping), two random seeds, and two model families (LightGBM and XGBoost) — four
base models in total. That run takes about 35-40 minutes on a CPU. This notebook runs the exact
same code at a smaller scale (3 folds, fewer trees, one seed) so it finishes in a few minutes and
you can see every step actually execute. The README has the full-scale numbers from the real run.

## 2.1 Setup

In [1]:
import time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

TARGET, ID = "Will_Buy_EV", "id"
N_FOLDS = 3          # production run: 10
N_TREES = 600        # production run: 6000 (early stopping usually stops well before that)
SEEDS = [42]         # production run: [42, 7]
INNER = 5            # inner folds for the nested target encoding — same at any scale

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
feat = [c for c in train.columns if c not in (ID, TARGET)]

def is_cat(s):
    return not pd.api.types.is_numeric_dtype(s)

y = (train[TARGET] == "Yes").astype(int).values
cats = [c for c in feat if is_cat(train[c])]
al = pd.concat([train[feat], test[feat]], ignore_index=True)   # train+test together, features only
n, n_te = len(train), len(test)
yfull = np.zeros(len(al)); yfull[:n] = y
print(f"train {train.shape}  test {test.shape}  categorical cols: {cats}")

train (668665, 15)  test (286571, 14)  categorical cols: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## 2.2 Nested target encoding — the leak-free way to use "what happened to similar rows before"

**The idea, in plain words:** replace a person's exact income value with "what fraction of people
with that same income bought an EV, in the training data." That's called *target encoding*, and
it's a direct way to hand the model the signal found in notebook 1.

**The trap:** if you compute that fraction using the row's *own* label, you've leaked the answer
into the feature — it's like a student grading their own exam using the answer key that includes
their own answers. The model would look perfect in testing and then fail on genuinely new data.

**The fix — nested (two-level) encoding:**
1. Split the training data into outer folds, same as ordinary cross-validation.
2. *Inside* each outer training fold, split again into inner folds. A row's encoding is always
   computed from the *other* inner folds — never from itself.
3. Validation and test rows (which are never part of any inner split) get their encoding from
   the *entire* outer training fold, since none of those rows contributed to their own number.

This way, every single encoded value a model ever sees was computed without that row's own label
touching it, at any level. The cross-validation score that comes out the other end is trustworthy.

In [1]:
def codes(s):
    return pd.factorize(s)[0]

KEYS = {
    "inc": codes(al.Annual_Income_USD), "com": codes(al.Daily_Commute_km),
    "inc1000": codes((al.Annual_Income_USD / 1000).round()),
    "inc250": codes((al.Annual_Income_USD / 250).round()),
    "com1": codes(al.Daily_Commute_km.round()),
    "com2": codes((al.Daily_Commute_km / 2).round()),
    "age": codes(al.Age),
}
TE_KEYS = tuple(KEYS.keys())
TE_M = (5, 20, 100)   # multiple smoothing strengths, blended together (see _te below)

def vc(s):
    return s.map(s.value_counts()).values

X_base = al[feat].copy()
for c in cats:
    X_base[c] = X_base[c].astype("category")
X_base["f_inc"] = vc(al.Annual_Income_USD)   # how over-produced each exact value is
X_base["f_com"] = vc(al.Daily_Commute_km)

def _te(k_tr, y_tr, k_ap, prior, m, ncode):
    """Smoothed target encoding: blend the group's own average with the overall
    average ('prior'), weighted by how many rows are in the group vs. the
    smoothing strength m. A group with only 2 rows gets pulled hard towards the
    prior; a group with 10,000 rows is barely smoothed at all."""
    s = np.bincount(k_tr, weights=y_tr, minlength=ncode)
    c = np.bincount(k_tr, minlength=ncode)
    return ((s + prior * m) / (c + m))[k_ap]

def te_features(train_idx, apply_idx, seed):
    """train_idx rows get out-of-fold encodings (inner K-fold, leak-free).
    apply_idx rows (validation + test) get encodings from ALL of train_idx."""
    out, prior = {}, yfull[train_idx].mean()
    for kn in TE_KEYS:
        k = KEYS[kn]
        kk = k.max() + 1
        for m in TE_M:
            col = np.zeros(len(al))
            for a, b in KFold(INNER, shuffle=True, random_state=seed).split(train_idx):
                col[train_idx[b]] = _te(k[train_idx[a]], yfull[train_idx[a]], k[train_idx[b]], prior, m, kk)
            col[apply_idx] = _te(k[train_idx], yfull[train_idx], k[apply_idx], prior, m, kk)
            out[f"te_{kn}_m{m}"] = col
    return pd.DataFrame(out)

print(f"target-encoding {len(TE_KEYS)} keys x {len(TE_M)} smoothing strengths = "
      f"{len(TE_KEYS) * len(TE_M)} engineered columns, rebuilt fresh inside every fold")

target-encoding 7 keys x 3 smoothing strengths = 21 engineered columns, rebuilt fresh inside every fold


Seven different ways of grouping income and commute (exact value, rounded to the nearest
\$1,000, rounded to the nearest \$250, and so on) times three smoothing strengths gives 21
engineered columns. Using several groupings together lets the model pick up both the hard exact
spikes (like \$30,000) and smoother nearby trends, without having to choose just one.

## 2.3 Train LightGBM and XGBoost inside cross-validation

Both models use **shallow trees** (`num_leaves=7` for LightGBM, `max_depth=4` for XGBoost) on
purpose. The target-encoded features are already strong, almost too strong — deep trees would
latch onto small-sample noise within them. Shallow trees plus a high `min_child_samples` /
`min_child_weight` force the model to only trust patterns backed by a lot of data, which tuning
experiments (see README) showed working clearly better than deeper, more "standard" trees.

In [1]:
def fit_lgb(Xt, yt, Xv, yv, seed):
    import lightgbm as lgb
    m = lgb.LGBMClassifier(
        n_estimators=N_TREES, learning_rate=0.05, num_leaves=7, min_child_samples=300,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_lambda=5,
        metric="auc", max_bin=1023, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xt, yt, eval_set=[(Xv, yv)], callbacks=[lgb.early_stopping(100, verbose=False)])
    return m.predict_proba(Xv)[:, 1], m.predict_proba(test_X)[:, 1], m.best_iteration_

def fit_xgb(Xt, yt, Xv, yv, seed):
    import xgboost as xgb
    m = xgb.XGBClassifier(
        n_estimators=N_TREES, learning_rate=0.05, max_depth=4, min_child_weight=10,
        subsample=0.8, colsample_bytree=0.4, reg_lambda=10, tree_method="hist",
        enable_categorical=True, eval_metric="auc", early_stopping_rounds=100,
        random_state=seed, n_jobs=-1)
    m.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
    return m.predict_proba(Xv)[:, 1], m.predict_proba(test_X)[:, 1], m.best_iteration

test_idx = np.arange(n, n + n_te)
results = {}
for seed in SEEDS:
    for model_name, fit_fn in [("lgb", fit_lgb), ("xgb", fit_xgb)]:
        t0 = time.time()
        oof = np.zeros(n)
        test_pred = np.zeros(n_te)
        for f, (tr_idx, va_idx) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(np.zeros(n), y)):
            T = te_features(tr_idx, np.concatenate([va_idx, test_idx]), seed + f)
            X = pd.concat([X_base, T], axis=1)
            test_X = X.iloc[test_idx]
            pv, pt, best_it = fit_fn(X.iloc[tr_idx], y[tr_idx], X.iloc[va_idx], y[va_idx], seed)
            oof[va_idx] = pv
            test_pred += pt / N_FOLDS
            print(f"[{model_name}_s{seed}] fold {f+1}/{N_FOLDS}  AUC {roc_auc_score(y[va_idx], pv):.5f}  "
                  f"best_iter {best_it}  {time.time()-t0:.0f}s", flush=True)
        auc = roc_auc_score(y, oof)
        print(f"[{model_name}_s{seed}] OOF AUC {auc:.5f}  ({time.time()-t0:.0f}s total)\n")
        results[f"{model_name}_s{seed}"] = {"oof": oof, "test": test_pred, "auc": auc}

np.savez("demo_run_predictions.npz",
         **{f"{k}_oof": v["oof"] for k, v in results.items()},
         **{f"{k}_test": v["test"] for k, v in results.items()})
print("saved demo_run_predictions.npz")
print({k: round(v["auc"], 5) for k, v in results.items()})

train (668665, 15)  test (286571, 14)  categorical cols: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
target-encoding 7 keys x 3 smoothing strengths = 21 engineered columns, rebuilt fresh inside every fold
[lgb_s42] fold 1/3  AUC 0.94470  best_iter 595  27s
[lgb_s42] fold 2/3  AUC 0.94611  best_iter 599  54s
[lgb_s42] fold 3/3  AUC 0.94577  best_iter 598  80s
[lgb_s42] OOF AUC 0.94552  (81s total)

[xgb_s42] fold 1/3  AUC 0.94474  best_iter 526  35s
[xgb_s42] fold 2/3  AUC 0.94613  best_iter 598  72s
[xgb_s42] fold 3/3  AUC 0.94583  best_iter 599  108s
[xgb_s42] OOF AUC 0.94556  (108s total)

saved demo_run_predictions.npz
{'lgb_s42': 0.94552, 'xgb_s42': 0.94556}


At this reduced demo scale (3 folds, 600 trees, one seed), both models land around
**0.9455**. Note `best_iter` is hitting the 600-tree cap in the later folds — the model hasn't
fully converged yet at this scale, which is expected; it's a speed/completeness trade-off for the
demo. The production run — 10 folds, up to 6,000 trees with early stopping, two seeds per model —
reached **OOF AUC 0.94608** on the full blend. Full numbers and the real training log are in the
README.

**Next:** `03_ensemble_and_final_submission.ipynb`